# DATATHON 2026 — Revenue & COGS Forecast v4
## LightGBM + XGBoost | Baseline-Residual | 2-Pass Forecast

### Kiến trúc tổng thể
```
Revenue = Baseline(seasonal × YoY growth) + Residual
                                               ↑
                              LightGBM + XGBoost học phần này
```

### Changelog v4
- ✅ Bỏ CatBoost (chậm, không đem lại nhiều giá trị)
- ✅ LightGBM đổi objective `huber` → `regression` (phù hợp với residuals 2 chiều)
- ✅ Thêm rolling mean/std của residuals làm features
- ✅ Xử lý NaN cẩn thận hơn (KNN-style ffill + median fallback)
- ✅ Bảng so sánh model đầy đủ
- ✅ COGS model riêng hoàn chỉnh

In [ ]:
# ═══════════════════════════════════════════════════════════
# 0. SETUP
# ═══════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import os, warnings, json
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from lunardate import LunarDate

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED    = 42
N_FOLDS = 5
np.random.seed(SEED)

TRAIN_START = pd.Timestamp('2012-07-04')
TRAIN_END   = pd.Timestamp('2022-12-31')
TEST_START  = pd.Timestamp('2023-01-01')
TEST_END    = pd.Timestamp('2024-07-01')

_cwd = os.path.abspath('')
DATA_PATH = (
    os.path.join(_cwd, 'data', 'processed')
    if os.path.isdir(os.path.join(_cwd, 'data'))
    else os.path.normpath(os.path.join(_cwd, '..', 'data', 'processed'))
)

def load(fname, **kw):
    path = os.path.join(DATA_PATH, fname)
    if not os.path.exists(path):
        print(f'⚠️  Không tìm thấy {fname} — bỏ qua')
        return None
    return pd.read_csv(path, **kw)

def compute_metrics(true, pred):
    """Tính MAE, RMSE, R² — tự xử lý inf/nan."""
    pred = np.where(np.isfinite(pred), pred, np.nanmedian(pred))
    return {
        'MAE' : mean_absolute_error(true, pred),
        'RMSE': np.sqrt(mean_squared_error(true, pred)),
        'R2'  : r2_score(true, pred),
    }

print(f'Train: {TRAIN_START.date()} → {TRAIN_END.date()}')
print(f'Test : {TEST_START.date()} → {TEST_END.date()}')
print('Setup xong ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. LOAD DATA
# ═══════════════════════════════════════════════════════════
sales       = load('sales.csv',             parse_dates=['Date'])
sample_sub  = load('sample_submission.csv', parse_dates=['Date'])
web_traffic = load('web_traffic.csv',       parse_dates=['date'])
orders      = load('orders.csv',            parse_dates=['order_date'])
order_items = load('order_items.csv')
promotions  = load('promotions.csv',        parse_dates=['start_date','end_date'])
inventory   = load('inventory.csv',         parse_dates=['snapshot_date'])

assert sales['Date'].max() < sample_sub['Date'].min(), '❌ LEAKAGE!'
test_dates = sample_sub[['Date']].sort_values('Date').reset_index(drop=True)

print(f'Train: {sales.shape} | {sales["Date"].min().date()} → {sales["Date"].max().date()}')
print(f'Test : {test_dates.shape} | {test_dates["Date"].min().date()} → {test_dates["Date"].max().date()}')
print('No leakage ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. BASELINE MODEL — Seasonal × YoY Growth
#
# Revenue = Baseline + Residual
# Baseline bắt trend dài hạn + seasonality
# Model ML chỉ học Residual → range nhỏ hơn → R² cao hơn
# ═══════════════════════════════════════════════════════════

def fit_baseline(df_train, upper_year):
    """
    Fit baseline trên data ≤ upper_year.
    Returns dict: seasonal profile + YoY growth rates.
    """
    tr = df_train[df_train['Date'].dt.year <= upper_year].copy()
    tr['year']  = tr['Date'].dt.year
    tr['month'] = tr['Date'].dt.month
    tr['day']   = tr['Date'].dt.day

    # Seasonal profile: chuẩn hoá theo mean năm
    ann = tr.groupby('year')[['Revenue','COGS']].transform('mean')
    tr['rev_norm']  = tr['Revenue'] / ann['Revenue']
    tr['cogs_norm'] = tr['COGS']    / ann['COGS']
    seasonal = tr.groupby(['month','day'])[['rev_norm','cogs_norm']].mean().reset_index()

    # YoY growth: geometric mean
    annual = tr.groupby('year')[['Revenue','COGS']].sum()
    if upper_year not in annual.index:
        upper_year = annual.index.max()
    avail = annual.index[annual.index <= upper_year]
    if len(avail) >= 2:
        full = annual.loc[avail]
        n    = len(full) - 1
        gr_rev  = (1 + full['Revenue'].pct_change().dropna()).prod() ** (1/n)
        gr_cogs = (1 + full['COGS'].pct_change().dropna()).prod() ** (1/n)
    else:
        gr_rev = gr_cogs = 1.0

    return {
        'seasonal'   : seasonal,
        'growth_rev' : gr_rev,   'growth_cogs': gr_cogs,
        'base_rev'   : annual.loc[upper_year,'Revenue'] / 365,
        'base_cogs'  : annual.loc[upper_year,'COGS']    / 365,
        'upper_year' : upper_year,
    }


def predict_baseline(dates_df, bp):
    """Dự báo baseline cho một tập ngày."""
    df = dates_df.copy()
    df['year']  = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['day']   = df['Date'].dt.day
    df = df.merge(bp['seasonal'], on=['month','day'], how='left')
    df['rev_norm']  = df['rev_norm'].fillna(1.0)
    df['cogs_norm'] = df['cogs_norm'].fillna(1.0)
    df['years_ahead'] = df['year'] - bp['upper_year']
    df['Revenue_pred'] = bp['base_rev']  * (bp['growth_rev']  ** df['years_ahead']) * df['rev_norm']
    df['COGS_pred']    = bp['base_cogs'] * (bp['growth_cogs'] ** df['years_ahead']) * df['cogs_norm']
    return df[['Date','Revenue_pred','COGS_pred']]


# ── Sanity check ──────────────────────────────────────────
bp_check = fit_baseline(sales, upper_year=2021)
val_22   = sales[sales['Date'].dt.year == 2022].copy()
pred_22  = predict_baseline(val_22[['Date']], bp_check)
val_22   = val_22.merge(pred_22, on='Date')
m = compute_metrics(val_22['Revenue'].values, val_22['Revenue_pred'].values)
print(f'Baseline sanity (fit≤2021, predict 2022):')
print(f'  Revenue MAE={m["MAE"]:,.0f} | RMSE={m["RMSE"]:,.0f} | R²={m["R2"]:.4f}')

# ── Tính baseline cho toàn train ─────────────────────────
min_yr = sales['Date'].dt.year.min()
max_yr = sales['Date'].dt.year.max()
base_preds = []
for yr in range(min_yr + 2, max_yr + 1):
    bp  = fit_baseline(sales, upper_year=yr - 1)
    val = sales[sales['Date'].dt.year == yr][['Date']].copy()
    base_preds.append(predict_baseline(val, bp))

base_df    = pd.concat(base_preds, ignore_index=True)
train_full = sales.merge(base_df, on='Date', how='inner')
train_full['rev_residual']  = train_full['Revenue'] - train_full['Revenue_pred']
train_full['cogs_residual'] = train_full['COGS']    - train_full['COGS_pred']

print(f'\nTrain full: {len(train_full):,} rows')
print(f'Rev  residual: mean={train_full["rev_residual"].mean():+,.0f} | std={train_full["rev_residual"].std():,.0f}')
print(f'COGS residual: mean={train_full["cogs_residual"].mean():+,.0f} | std={train_full["cogs_residual"].std():,.0f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. ĐỊNH NGHĨA SỰ KIỆN
# ═══════════════════════════════════════════════════════════
MIN_DATE = sales['Date'].min()

# Tết Nguyên Đán (lunardate → solar)
tet_map        = {y: pd.Timestamp(LunarDate(y,1,1).toSolarDate()) for y in range(2012,2027)}
tet_dates_list = sorted(tet_map.values())

# MegaSale: 9.9, 10.10, 11.11, 12.12
MEGA_SALE_DAYS = {(9,9),(10,10),(11,11),(12,12)}
mega_dates     = sorted([
    pd.Timestamp(year=y, month=m, day=d)
    for y in range(2012,2026) for m,d in MEGA_SALE_DAYS
])

# Ngày lễ VN
VN_HOLIDAYS = {(1,1),(4,30),(5,1),(9,2),(10,20),(3,8),(11,20)}

def days_to_next(date, events):
    future = [e for e in events if e >= date]
    return (future[0] - date).days if future else 999

def days_since_last(date, events):
    past = [e for e in events if e <= date]
    return (date - past[-1]).days if past else 999

print(f'Tet 2023: {tet_map[2023].date()} | Tet 2024: {tet_map[2024].date()}')
print(f'MegaSale events: {len(mega_dates)} ngày')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════

def add_calendar_features(df):
    """Calendar + Fourier encoding."""
    df['year']           = df['Date'].dt.year
    df['month']          = df['Date'].dt.month
    df['day']            = df['Date'].dt.day
    df['dayofweek']      = df['Date'].dt.dayofweek
    df['dayofyear']      = df['Date'].dt.dayofyear
    df['quarter']        = df['Date'].dt.quarter
    df['is_weekend']     = (df['dayofweek'] >= 5).astype(int)
    df['is_month_end']   = df['Date'].dt.is_month_end.astype(int)
    df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
    df['is_quarter_end'] = df['Date'].dt.is_quarter_end.astype(int)
    df['is_vn_holiday']  = df['Date'].apply(lambda d: int((d.month,d.day) in VN_HOLIDAYS))
    df['time_idx']       = (df['Date'] - MIN_DATE).dt.days
    df['year_idx']       = (df['Date'] - TRAIN_START).dt.days / 365.25
    df['year_idx_sq']    = df['year_idx'] ** 2
    # Fourier
    doy = df['dayofyear']; dow = df['dayofweek']; mo = df['month']
    for k in range(1,4):
        df[f'sin_year_{k}'] = np.sin(2*np.pi*k*doy/365.25)
        df[f'cos_year_{k}'] = np.cos(2*np.pi*k*doy/365.25)
    for k in range(1,3):
        df[f'sin_week_{k}'] = np.sin(2*np.pi*k*dow/7)
        df[f'cos_week_{k}'] = np.cos(2*np.pi*k*dow/7)
    df['month_sin'] = np.sin(2*np.pi*mo/12)
    df['month_cos'] = np.cos(2*np.pi*mo/12)
    return df


def add_tet_features(df):
    """
    3 phases Tết:
    Sắm Tết (-20→-1): Revenue TĂNG
    Trong Tết (0→5) : Revenue GIẢM mạnh
    Sau Tết (6→14)  : Phục hồi
    """
    df['days_to_tet']    = df['Date'].apply(lambda d: days_to_next(d, tet_dates_list)).clip(0,90)
    df['days_since_tet'] = df['Date'].apply(lambda d: days_since_last(d, tet_dates_list)).clip(0,90)
    df['is_sam_tet']     = ((df['days_to_tet']>=1)    & (df['days_to_tet']<=20)).astype(int)
    df['is_tet_core']    = (df['days_to_tet']==0).astype(int)
    df['is_trong_tet']   = ((df['days_since_tet']>=1) & (df['days_since_tet']<=5)).astype(int)
    df['is_sau_tet']     = ((df['days_since_tet']>=6) & (df['days_since_tet']<=14)).astype(int)
    df['tet_proximity']  = np.where(df['days_to_tet']<=20, 1/(df['days_to_tet']+1), 0)
    return df


def add_megasale_features(df):
    """
    3 phases MegaSale:
    Pre (2 ngày trước): nhịn mua → revenue GIẢ thấp
    Sale day           : spike đỉnh
    Post (1 ngày sau)  : vẫn cao
    """
    df['days_to_next_mega']    = df['Date'].apply(lambda d: days_to_next(d,mega_dates)).clip(0,90)
    df['days_since_mega']      = df['Date'].apply(lambda d: days_since_last(d,mega_dates)).clip(0,30)
    df['is_mega_sale']         = (df['days_to_next_mega']==0).astype(int)
    df['is_pre_sale_suppress'] = ((df['days_to_next_mega']>=1)&(df['days_to_next_mega']<=2)).astype(int)
    df['is_post_sale_bounce']  = ((df['days_since_mega']>=1) &(df['days_since_mega']<=1)).astype(int)
    df['mega_countdown']       = np.where(df['days_to_next_mega']<=7, 7-df['days_to_next_mega'], 0).astype(float)
    df['is_pre_mega_3d']       = (df['days_to_next_mega']<=3).astype(int)
    return df


def add_inventory_features(df):
    """Stockout signal: phân biệt hết hàng vs giảm nhu cầu thực."""
    if 'inv_fill_rate' not in df.columns:
        return df
    fill_med  = df['inv_fill_rate'].median()
    supply_lo = df['inv_days_supply'].quantile(0.25) if 'inv_days_supply' in df.columns else 7
    df['stockout_signal']     = ((df['inv_fill_rate'] < fill_med) &
                                  (df.get('inv_days_supply', pd.Series(999, index=df.index)) < supply_lo)).astype(int)
    df['inv_fill_x_stockout'] = df['inv_fill_rate'] * df['stockout_signal']
    if 'inv_days_supply' in df.columns:
        df['log_inv_days_supply'] = np.log1p(df['inv_days_supply'])
    return df


def add_all_features(df):
    """Pipeline tổng hợp — gọi theo thứ tự cố định."""
    df = df.sort_values('Date').reset_index(drop=True)
    df = add_calendar_features(df)
    df = add_tet_features(df)
    df = add_megasale_features(df)
    df = add_inventory_features(df)
    df = df.loc[:, ~df.columns.duplicated()]
    return df

print('Feature functions ready ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. PROMO DAILY
# Test period: replicate promo từ 2 năm trước (không fillna(0))
# ═══════════════════════════════════════════════════════════

def build_promo_daily(promotions_df, target_date_range):
    if promotions_df is None:
        return pd.DataFrame(columns=['Date','has_promo','promo_discount',
                                      'promo_type_perc','promo_intensity'])
    target_min = pd.Timestamp(target_date_range.min())
    target_max = pd.Timestamp(target_date_range.max())

    if target_min.year >= 2023:
        # Replicate từ 2 năm trước
        rep_start = target_min - pd.DateOffset(years=2)
        rep_end   = target_max - pd.DateOffset(years=2)
        src = promotions_df[
            (promotions_df['start_date'] >= rep_start) &
            (promotions_df['end_date']   <= rep_end)
        ].copy()
        src['start_date'] = src['start_date'] + pd.DateOffset(years=2)
        src['end_date']   = src['end_date']   + pd.DateOffset(years=2)
    else:
        src = promotions_df[
            (promotions_df['end_date']   >= target_min) &
            (promotions_df['start_date'] <= target_max)
        ].copy()

    rows = []
    for _, row in src.iterrows():
        for d in pd.date_range(row['start_date'], row['end_date']):
            if target_min <= d <= target_max:
                rows.append({
                    'Date'          : d,
                    'has_promo'     : 1,
                    'promo_discount': row.get('discount_value', 0),
                    'promo_type_perc': 1 if row.get('promo_type','') == 'percentage' else 0,
                })
    if not rows:
        return pd.DataFrame(columns=['Date','has_promo','promo_discount',
                                      'promo_type_perc','promo_intensity'])
    pl  = pd.DataFrame(rows)
    ity = pl.groupby('Date').size().reset_index(name='promo_intensity')
    agg = pl.groupby('Date').agg(
        has_promo       = ('has_promo','max'),
        promo_discount  = ('promo_discount','max'),
        promo_type_perc = ('promo_type_perc','max'),
    ).reset_index()
    return agg.merge(ity, on='Date', how='left')


PROMO_COLS  = ['has_promo','promo_discount','promo_type_perc','promo_intensity']
train_promo = build_promo_daily(promotions, sales['Date'])
test_promo  = build_promo_daily(promotions, test_dates['Date'])

train_full = train_full.merge(train_promo, on='Date', how='left')
for c in PROMO_COLS:
    train_full[c] = train_full[c].fillna(0)

print(f'Train promo: {len(train_promo)} ngày ({len(train_promo)/len(sales)*100:.1f}%)')
print(f'Test  promo: {len(test_promo)} ngày ({len(test_promo)/len(test_dates)*100:.1f}%)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. EXTERNAL FEATURES
# ═══════════════════════════════════════════════════════════

# ── Web Traffic: lag365 (no leakage) ──────────────────────
WT_LAG_COLS = []
if web_traffic is not None:
    wt = web_traffic.rename(columns={'date':'Date'})
    agg_wt = {c:'sum'  for c in ['sessions','unique_visitors','page_views'] if c in wt.columns}
    agg_wt.update({c:'mean' for c in ['bounce_rate','avg_session_duration_sec'] if c in wt.columns})
    wt_daily = wt.groupby('Date').agg(agg_wt).reset_index().sort_values('Date').reset_index(drop=True)
    for col in [c for c in wt_daily.columns if c != 'Date']:
        wt_daily[f'{col}_lag365'] = wt_daily[col].shift(365)
    WT_LAG_COLS  = [c for c in wt_daily.columns if c.endswith('_lag365')]
    web_features = wt_daily[['Date'] + WT_LAG_COLS]
    train_full   = train_full.merge(web_features, on='Date', how='left')
    print(f'Web traffic lag365: {WT_LAG_COLS}')

# ── Inventory ──────────────────────────────────────────────
INV_COLS = []
if inventory is not None:
    inv_agg = inventory.groupby('snapshot_date').agg(
        inv_days_supply   = ('days_of_supply','mean'),
        inv_fill_rate     = ('fill_rate','mean'),
        inv_stockout_days = ('stockout_days','mean'),
        inv_stockout_flag = ('stockout_flag','max'),
    ).reset_index().rename(columns={'snapshot_date':'Date'})
    full_range = pd.DataFrame({'Date': pd.date_range(TRAIN_START, TEST_END)})
    inv_daily  = full_range.merge(inv_agg, on='Date', how='left')
    INV_COLS   = ['inv_days_supply','inv_fill_rate','inv_stockout_days','inv_stockout_flag']
    inv_daily[INV_COLS] = inv_daily[INV_COLS].ffill().bfill()
    train_full = train_full.merge(inv_daily[['Date']+INV_COLS], on='Date', how='left')
    print(f'Inventory cols: {INV_COLS}')

print(f'train_full shape: {train_full.shape}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. APPLY FEATURES + LAG/ROLLING RESIDUALS
# ═══════════════════════════════════════════════════════════
train_full = add_all_features(train_full)
train_full = train_full.sort_values('Date').reset_index(drop=True)

# ── Lag residuals ──────────────────────────────────────────
# lag365: cùng ngày năm trước
# lag372: cùng ngày CÙNG THỨ năm trước (= 365 + 7)
train_full['rev_lag_365']  = train_full['rev_residual'].shift(365)
train_full['rev_lag_372']  = train_full['rev_residual'].shift(372)
train_full['cogs_lag_365'] = train_full['cogs_residual'].shift(365)
train_full['cogs_lag_372'] = train_full['cogs_residual'].shift(372)

# ── Rolling mean/std của residuals ─────────────────────────
# shift(1) trước rolling để không có lookahead
# Rolling trên residuals giúp model nắm được momentum ngắn hạn
s_rev  = train_full['rev_residual'].shift(1)
s_cogs = train_full['cogs_residual'].shift(1)
for w in [7, 30]:
    train_full[f'rev_roll_mean_{w}']  = s_rev.rolling(w).mean()
    train_full[f'rev_roll_std_{w}']   = s_rev.rolling(w).std()
    train_full[f'cogs_roll_mean_{w}'] = s_cogs.rolling(w).mean()
    train_full[f'cogs_roll_std_{w}']  = s_cogs.rolling(w).std()

# ── Monthly mean lag365 ────────────────────────────────────
train_full['ym'] = train_full['Date'].dt.strftime('%Y-%m')
ym_means = train_full.groupby('ym')[['Revenue','COGS']].mean().reset_index()
ym_means.columns = ['ym','rev_monthly_mean','cogs_monthly_mean']
train_full = train_full.merge(ym_means, on='ym', how='left')
train_full['rev_monthly_mean_lag365']  = train_full['rev_monthly_mean'].shift(365)
train_full['cogs_monthly_mean_lag365'] = train_full['cogs_monthly_mean'].shift(365)

# ── Interaction features ───────────────────────────────────
train_full['sam_tet_x_promo'] = train_full['is_sam_tet']   * train_full['has_promo']
train_full['mega_x_promo']    = train_full['is_mega_sale']  * train_full['has_promo']
train_full['weekend_x_promo'] = train_full['is_weekend']    * train_full['has_promo']

print(f'train_full sau features: {train_full.shape}')
for c in ['rev_lag_365','rev_lag_372','rev_monthly_mean_lag365',
           'rev_roll_mean_7','rev_roll_mean_30']:
    print(f'  NaN {c}: {train_full[c].isna().sum()}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 8. ĐỊNH NGHĨA FEATURE SETS
# ═══════════════════════════════════════════════════════════

TIME_FEATS = [
    'year','month','day','dayofweek','dayofyear','quarter',
    'is_month_end','is_month_start','is_quarter_end',
    'time_idx','year_idx','year_idx_sq','is_vn_holiday',
    'sin_year_1','cos_year_1','sin_year_2','cos_year_2','sin_year_3','cos_year_3',
    'sin_week_1','cos_week_1','sin_week_2','cos_week_2',
    'month_sin','month_cos',
]

TET_FEATS = [
    'days_to_tet','days_since_tet',
    'is_sam_tet','is_tet_core','is_trong_tet','is_sau_tet','tet_proximity',
]

MEGA_FEATS = [
    'days_to_next_mega','days_since_mega',
    'is_mega_sale','is_pre_sale_suppress','is_post_sale_bounce',
    'mega_countdown','is_pre_mega_3d',
]

PROMO_FEATS    = PROMO_COLS.copy()
INTERACT_FEATS = ['sam_tet_x_promo','mega_x_promo','weekend_x_promo']
INV_FEATS      = INV_COLS + (
    ['stockout_signal','inv_fill_x_stockout','log_inv_days_supply']
    if 'stockout_signal' in train_full.columns else []
)
WEB_FEATS = [c for c in WT_LAG_COLS if c in train_full.columns]

BASE_FEATS = TIME_FEATS + TET_FEATS + MEGA_FEATS + PROMO_FEATS + INTERACT_FEATS + INV_FEATS + WEB_FEATS

# Baseline + lag/rolling đặc biệt cho từng target
REV_LAG_FEATS  = [
    'Revenue_pred',
    'rev_lag_365','rev_lag_372',
    'rev_monthly_mean_lag365',
    'rev_roll_mean_7','rev_roll_std_7',
    'rev_roll_mean_30','rev_roll_std_30',
]
COGS_LAG_FEATS = [
    'COGS_pred',
    'cogs_lag_365','cogs_lag_372',
    'cogs_monthly_mean_lag365',
    'cogs_roll_mean_7','cogs_roll_std_7',
    'cogs_roll_mean_30','cogs_roll_std_30',
]

REV_FEATS  = [c for c in BASE_FEATS + REV_LAG_FEATS  if c in train_full.columns]
COGS_FEATS = [c for c in BASE_FEATS + COGS_LAG_FEATS if c in train_full.columns]

# Drop NaN
ml_rev  = train_full.dropna(subset=REV_FEATS  + ['rev_residual']).reset_index(drop=True)
ml_cogs = train_full.dropna(subset=COGS_FEATS + ['cogs_residual']).reset_index(drop=True)

assert ml_rev[REV_FEATS].isna().sum().sum()   == 0, '❌ NaN trong REV_FEATS'
assert ml_cogs[COGS_FEATS].isna().sum().sum() == 0, '❌ NaN trong COGS_FEATS'

print(f'Revenue  features: {len(REV_FEATS)}')
print(f'COGS     features: {len(COGS_FEATS)}')
print(f'ml_rev  rows: {len(ml_rev):,} | {ml_rev["Date"].dt.year.min()} → {ml_rev["Date"].dt.year.max()}')
print(f'ml_cogs rows: {len(ml_cogs):,}')
print('No NaN ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 9. CV SETUP — Expanding window theo năm
# ═══════════════════════════════════════════════════════════

def get_cv_splits(df, val_years=[2020,2021,2022]):
    splits = []
    for vy in val_years:
        tr = df[df['year'] < vy].index
        va = df[df['year'] == vy].index
        if len(tr) > 0 and len(va) > 0:
            splits.append((tr, va, vy))
    return splits

cv_rev  = get_cv_splits(ml_rev)
cv_cogs = get_cv_splits(ml_cogs)

print('CV splits Revenue:')
for ti,vi,vy in cv_rev:
    print(f'  val={vy}: train={len(ti):,} | val={len(vi):,}')


def cv_evaluate(make_model_fn, df, feat_cols, target_resid,
                baseline_col, actual_col, cv_splits, name='Model'):
    """CV evaluation — trả về metrics DataFrame + fold predictions."""
    rows, fold_preds = [], {}
    for ti, vi, vy in cv_splits:
        m = make_model_fn()
        m.fit(df.loc[ti, feat_cols], df.loc[ti, target_resid])
        resid = np.clip(m.predict(df.loc[vi, feat_cols]), -1e9, 1e9)
        bl_v  = df.loc[vi, baseline_col].values
        ac_v  = df.loc[vi, actual_col].values
        final = bl_v + resid
        met   = compute_metrics(ac_v, final)
        met['fold'] = vy
        rows.append(met)
        fold_preds[vy] = (final, ac_v, resid, bl_v)

    df_m = pd.DataFrame(rows)
    avg  = df_m[['MAE','RMSE','R2']].mean()
    print(f'\n{name}:')
    for _, r in df_m.iterrows():
        print(f'  fold={int(r["fold"])} | MAE={r["MAE"]:>10,.0f} | RMSE={r["RMSE"]:>10,.0f} | R²={r["R2"]:.4f}')
    print(f'  AVG   | MAE={avg["MAE"]:>10,.0f} | RMSE={avg["RMSE"]:>10,.0f} | R²={avg["R2"]:.4f}')
    return df_m, fold_preds

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10. OPTUNA TUNING — LightGBM + XGBoost
#
# LightGBM: objective='regression' (MSE)
# Lý do: residuals có phân phối 2 chiều (âm + dương)
# Huber chỉ phù hợp khi target lớn dương (như log_revenue)
# ═══════════════════════════════════════════════════════════

def make_optuna_objective(model_type, df, feat_cols, target_resid,
                           baseline_col, actual_col, cv_splits):
    def objective(trial):
        if model_type == 'lgbm':
            params = {
                'objective'         : 'regression',   # MSE cho residuals 2 chiều
                'n_estimators'      : 2000,
                'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
                'num_leaves'        : trial.suggest_int('num_leaves', 20, 80),
                'max_depth'         : trial.suggest_int('max_depth', 4, 8),
                'min_child_samples' : trial.suggest_int('min_child_samples', 30, 120),
                'subsample'         : trial.suggest_float('subsample', 0.6, 0.9),
                'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.6, 0.9),
                'reg_alpha'         : trial.suggest_float('reg_alpha', 0.05, 8.0, log=True),
                'reg_lambda'        : trial.suggest_float('reg_lambda', 1.0, 20.0, log=True),
                'subsample_freq'    : 1,
                'random_state'      : SEED, 'n_jobs': -1, 'verbose': -1,
            }
            model = lgb.LGBMRegressor(**params)
        elif model_type == 'xgboost':
            params = {
                'objective'            : 'reg:squarederror',
                'n_estimators'         : 2000,
                'early_stopping_rounds': 50,
                'learning_rate'        : trial.suggest_float('learning_rate', 0.003, 0.05, log=True),
                'max_depth'            : trial.suggest_int('max_depth', 3, 8),
                'min_child_weight'     : trial.suggest_int('min_child_weight', 20, 150),
                'subsample'            : trial.suggest_float('subsample', 0.6, 0.95),
                'colsample_bytree'     : trial.suggest_float('colsample_bytree', 0.6, 0.95),
                'gamma'                : trial.suggest_float('gamma', 0.0, 3.0),
                'reg_alpha'            : trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
                'reg_lambda'           : trial.suggest_float('reg_lambda', 1.0, 20.0, log=True),
                'tree_method'          : 'hist',
                'random_state'         : SEED, 'n_jobs': -1, 'verbosity': 0,
            }
            model = xgb.XGBRegressor(**params)

        scores = []
        for ti, vi, vy in cv_splits[-2:]:   # 2 fold cuối đủ nhanh
            try:
                if model_type == 'xgboost':
                    model.fit(df.loc[ti, feat_cols], df.loc[ti, target_resid],
                              eval_set=[(df.loc[vi, feat_cols], df.loc[vi, target_resid])],
                              verbose=False)
                else:
                    model.fit(df.loc[ti, feat_cols], df.loc[ti, target_resid])
                resid = np.clip(model.predict(df.loc[vi, feat_cols]), -1e9, 1e9)
                final = df.loc[vi, baseline_col].values + resid
                scores.append(mean_absolute_error(df.loc[vi, actual_col].values, final))
            except Exception:
                return float('inf')
        return float(np.mean(scores)) if scores else float('inf')
    return objective


N_TRIALS = 50
studies  = {}

for target_name, df_ml, feat_cols, target_resid, baseline_col, actual_col in [
    ('rev',  ml_rev,  REV_FEATS,  'rev_residual',  'Revenue_pred', 'Revenue'),
    ('cogs', ml_cogs, COGS_FEATS, 'cogs_residual', 'COGS_pred',    'COGS'),
]:
    cv_splits = cv_rev if target_name == 'rev' else cv_cogs
    for mtype in ['lgbm', 'xgboost']:
        key = f'{mtype}_{target_name}'
        print(f'Tuning {key} ({N_TRIALS} trials)...')
        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=SEED)
        )
        study.optimize(
            make_optuna_objective(mtype, df_ml, feat_cols, target_resid,
                                   baseline_col, actual_col, cv_splits),
            n_trials=N_TRIALS, show_progress_bar=True
        )
        studies[key] = study
        print(f'  Best MAE: {study.best_value:,.0f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 11. BUILD BEST PARAMS & FINAL MODELS
# ═══════════════════════════════════════════════════════════

def build_params(model_type, study):
    bp = study.best_params.copy()
    if model_type == 'lgbm':
        return {
            'objective'     : 'regression',
            'n_estimators'  : 3000,
            'subsample_freq': 1,
            'random_state'  : SEED, 'n_jobs': -1, 'verbose': -1,
            **bp
        }
    elif model_type == 'xgboost':
        return {
            'objective'  : 'reg:squarederror',
            'n_estimators': 3000,
            'tree_method': 'hist',
            'random_state': SEED, 'n_jobs': -1, 'verbosity': 0,
            **bp
        }

def make_model(model_type, params):
    if model_type == 'lgbm'   : return lgb.LGBMRegressor(**params)
    if model_type == 'xgboost': return xgb.XGBRegressor(**params)

best_params = {key: build_params(key.split('_')[0], study)
               for key, study in studies.items()}

print('Best params summary:')
for k, p in best_params.items():
    n = p.get('n_estimators', '?')
    lr = p.get('learning_rate', '?')
    print(f'  {k}: n_estimators={n} | lr={lr:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 12. CV EVALUATION
# ═══════════════════════════════════════════════════════════
print('=== CV REVENUE ===')
fp_rev = {}
for mtype in ['lgbm','xgboost']:
    _, preds = cv_evaluate(
        lambda mt=mtype: make_model(mt, best_params[f'{mt}_rev']),
        ml_rev, REV_FEATS, 'rev_residual', 'Revenue_pred', 'Revenue',
        cv_rev, f'{mtype}_rev'
    )
    fp_rev[mtype] = preds

print('\n=== CV COGS ===')
fp_cogs = {}
for mtype in ['lgbm','xgboost']:
    _, preds = cv_evaluate(
        lambda mt=mtype: make_model(mt, best_params[f'{mt}_cogs']),
        ml_cogs, COGS_FEATS, 'cogs_residual', 'COGS_pred', 'COGS',
        cv_cogs, f'{mtype}_cogs'
    )
    fp_cogs[mtype] = preds

In [ ]:
# ═══════════════════════════════════════════════════════════
# 13. HILL CLIMBING ENSEMBLE
# ═══════════════════════════════════════════════════════════

def hill_climbing_ensemble(resid_preds, baselines, y_true,
                            n_rounds=1000, step=0.05, seed=SEED):
    """
    Tối ưu trọng số ensemble bằng composite metric:
    0.4*R² + 0.3*MAE_norm + 0.3*RMSE_norm
    R² được ưu tiên cao nhất (đề thi yêu cầu).
    """
    np.random.seed(seed)
    n = len(resid_preds)
    w = np.ones(n) / n

    def score(weights):
        blended = baselines + sum(r*wi for r,wi in zip(resid_preds, weights))
        mae  = mean_absolute_error(y_true, blended)
        rmse = np.sqrt(mean_squared_error(y_true, blended))
        r2   = r2_score(y_true, blended)
        sc   = max(np.abs(y_true).mean(), 1)
        return 0.3*(mae/sc) + 0.3*(rmse/sc) + 0.4*(1-r2)

    best = score(w)
    for _ in range(n_rounds):
        improved = False
        for i in np.random.permutation(n):
            nw = w.copy(); nw[i] += step; nw /= nw.sum()
            s = score(nw)
            if s < best:
                best, w, improved = s, nw, True
        if not improved and step > 0.001:
            step *= 0.9
    return w


# Fold cuối gần test nhất → dùng để tìm weights
last_vy = cv_rev[-1][2]

# Revenue
resids_rev = [fp_rev[m][last_vy][2] for m in ['lgbm','xgboost']]
bl_rev_val = fp_rev['lgbm'][last_vy][3]
ac_rev_val = fp_rev['lgbm'][last_vy][1]
W_REV = hill_climbing_ensemble(resids_rev, bl_rev_val, ac_rev_val)

ens_rev_pred  = bl_rev_val + sum(r*w for r,w in zip(resids_rev, W_REV))
m_ens_rev     = compute_metrics(ac_rev_val, ens_rev_pred)

# COGS
last_vy_c = cv_cogs[-1][2]
resids_cogs = [fp_cogs[m][last_vy_c][2] for m in ['lgbm','xgboost']]
bl_cogs_val = fp_cogs['lgbm'][last_vy_c][3]
ac_cogs_val = fp_cogs['lgbm'][last_vy_c][1]
W_COGS = hill_climbing_ensemble(resids_cogs, bl_cogs_val, ac_cogs_val)

ens_cogs_pred = bl_cogs_val + sum(r*w for r,w in zip(resids_cogs, W_COGS))
m_ens_cogs    = compute_metrics(ac_cogs_val, ens_cogs_pred)

print(f'Revenue  weights: LGBM={W_REV[0]:.3f}  | XGB={W_REV[1]:.3f}')
print(f'COGS     weights: LGBM={W_COGS[0]:.3f} | XGB={W_COGS[1]:.3f}')
print(f'\nEnsemble Revenue  val={last_vy}:  MAE={m_ens_rev["MAE"]:,.0f} | R²={m_ens_rev["R2"]:.4f}')
print(f'Ensemble COGS     val={last_vy_c}: MAE={m_ens_cogs["MAE"]:,.0f} | R²={m_ens_cogs["R2"]:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 14. BẢNG SO SÁNH MÔ HÌNH
# ═══════════════════════════════════════════════════════════
rows = []

# Baseline only
for tname, df_ml, bl_col, ac_col, cv_sp in [
    ('Revenue', ml_rev,  'Revenue_pred', 'Revenue', cv_rev),
    ('COGS',    ml_cogs, 'COGS_pred',    'COGS',    cv_cogs),
]:
    ms = [compute_metrics(df_ml.loc[vi, ac_col].values, df_ml.loc[vi, bl_col].values)
          for ti,vi,vy in cv_sp]
    rows.append({'Target': tname, 'Model': 'Baseline only',
                 'MAE': np.mean([m['MAE'] for m in ms]),
                 'R2' : np.mean([m['R2']  for m in ms])})

# Từng model
for tname, fp, df_ml, cv_sp in [
    ('Revenue', fp_rev,  ml_rev,  cv_rev),
    ('COGS',    fp_cogs, ml_cogs, cv_cogs),
]:
    for mtype in ['lgbm','xgboost']:
        ms = [compute_metrics(fp[mtype][vy][1], fp[mtype][vy][0]) for _,_,vy in cv_sp]
        rows.append({'Target': tname,
                     'Model' : {'lgbm':'LightGBM','xgboost':'XGBoost'}[mtype],
                     'MAE'   : np.mean([m['MAE'] for m in ms]),
                     'R2'    : np.mean([m['R2']  for m in ms])})

# Ensemble
rows.append({'Target':'Revenue','Model':'Ensemble (HC)','MAE':m_ens_rev['MAE'], 'R2':m_ens_rev['R2']})
rows.append({'Target':'COGS',   'Model':'Ensemble (HC)','MAE':m_ens_cogs['MAE'],'R2':m_ens_cogs['R2']})

df_compare = pd.DataFrame(rows)

print('=' * 62)
print('SO SÁNH MÔ HÌNH — CV Mean (2020/2021/2022)')
print('=' * 62)
for target in ['Revenue','COGS']:
    sub = df_compare[df_compare['Target'] == target]
    best_mae = sub['MAE'].min()
    best_r2  = sub['R2'].max()
    print(f'\n  {target}:')
    print(f'  {"Model":<18} | {"MAE":>12} | {"R²":>7}')
    print(f'  {"-"*18}-+-{"-"*12}-+-{"-"*7}')
    for _, r in sub.iterrows():
        tag = ' ✅' if r['R2'] == best_r2 else ('  ⬇️ MAE' if r['MAE'] == best_mae else '')
        print(f'  {r["Model"]:<18} | {r["MAE"]:>12,.0f} | {r["R2"]:>7.4f}{tag}')

df_compare.to_csv('metrics_comparison.csv', index=False)
print('\nmetrics_comparison.csv saved ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 15. FINAL MODEL TRAINING
# ═══════════════════════════════════════════════════════════
print('Training final models trên toàn bộ train...')
final_models = {}
for tname, df_ml, feat_cols, target_resid in [
    ('rev',  ml_rev,  REV_FEATS,  'rev_residual'),
    ('cogs', ml_cogs, COGS_FEATS, 'cogs_residual'),
]:
    for mtype in ['lgbm','xgboost']:
        key = f'{mtype}_{tname}'
        print(f'  {key}...')
        m = make_model(mtype, best_params[key])
        m.fit(df_ml[feat_cols], df_ml[target_resid])
        final_models[key] = m
print('Done ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 16. SHAP INTERPRETABILITY
# ═══════════════════════════════════════════════════════════
sample_idx = np.random.choice(len(ml_rev), min(800, len(ml_rev)), replace=False)
X_shap     = ml_rev.iloc[sample_idx][REV_FEATS]

explainer = shap.TreeExplainer(final_models['lgbm_rev'])
shap_vals = explainer(X_shap)
mean_shap = pd.Series(np.abs(shap_vals.values).mean(0), index=REV_FEATS).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
plt.sca(axes[0])
shap.summary_plot(shap_vals, X_shap, show=False, max_display=20)
axes[0].set_title('SHAP Summary — Revenue (LightGBM)', fontweight='bold')

top20  = mean_shap.head(20)
colors = ['#E74C3C' if any(kw in c for kw in ['tet','mega','sam','promo','stockout','vn'])
          else '#3498DB' for c in top20.index[::-1]]
axes[1].barh(top20.index[::-1], top20.values[::-1], color=colors)
axes[1].set_title('Mean |SHAP| — Top 20', fontweight='bold')
axes[1].set_xlabel('Mean |SHAP|')
from matplotlib.patches import Patch
axes[1].legend(handles=[
    Patch(color='#E74C3C', label='VN / Event features'),
    Patch(color='#3498DB', label='General features'),
], fontsize=9)
plt.tight_layout()
plt.savefig('shap_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

biz = {
    'Revenue_pred'            : 'Baseline seasonal × YoY — signal mạnh nhất',
    'rev_lag_365'             : 'Residual cùng ngày năm trước',
    'rev_lag_372'             : 'Residual cùng thứ năm trước (+7)',
    'rev_monthly_mean_lag365' : 'Trung bình tháng cùng kỳ năm trước',
    'rev_roll_mean_7'         : 'Momentum 7 ngày gần nhất',
    'rev_roll_mean_30'        : 'Trend 30 ngày gần nhất',
    'days_to_tet'             : 'Khoảng cách Tết → spike Sắm Tết',
    'is_sam_tet'              : 'Giai đoạn Sắm Tết (-20 đến -1)',
    'is_mega_sale'            : 'Ngày 11.11/12.12 — spike lớn nhất',
    'is_pre_sale_suppress'    : 'Nhịn mua 2 ngày trước MegaSale',
    'stockout_signal'         : 'Hết hàng — revenue thấp do supply',
    'time_idx'                : 'Trend tuyến tính dài hạn',
}
print('\nTop 10 features (Revenue):')
for i,(f,v) in enumerate(mean_shap.head(10).items(), 1):
    print(f'  {i:2d}. {f:35s} | SHAP={v:.4f} | {biz.get(f,"—")}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 17. 2-PASS FORECAST
# Pass 1: 2023 dùng lag từ 2022 ACTUALS
# Pass 2: 2024 dùng lag từ 2023 PREDICTIONS
# ═══════════════════════════════════════════════════════════

# Baseline test
bp_final      = fit_baseline(sales, upper_year=2022)
test_baseline = predict_baseline(test_dates[['Date']], bp_final)

# Build test features
test_full = test_dates.merge(test_baseline, on='Date', how='left')
test_full = add_all_features(test_full)

# Promo (replicated)
test_full = test_full.merge(test_promo, on='Date', how='left')
for c in PROMO_COLS:
    test_full[c] = test_full[c].fillna(0)
test_full['sam_tet_x_promo'] = test_full['is_sam_tet']   * test_full['has_promo']
test_full['mega_x_promo']    = test_full['is_mega_sale']  * test_full['has_promo']
test_full['weekend_x_promo'] = test_full['is_weekend']    * test_full['has_promo']

# Web traffic lag365
if WT_LAG_COLS:
    test_full = test_full.merge(wt_daily[['Date']+WT_LAG_COLS], on='Date', how='left')

# Inventory
if INV_COLS:
    test_full = test_full.merge(inv_daily[['Date']+INV_COLS], on='Date', how='left')
    test_full[INV_COLS] = test_full[INV_COLS].ffill().bfill()
    test_full = add_inventory_features(test_full)

# Lookup dicts từ train
rev_resid_map  = dict(zip(train_full['Date'], train_full['rev_residual']))
cogs_resid_map = dict(zip(train_full['Date'], train_full['cogs_residual']))
ym_rev_lookup  = train_full.groupby('ym')['Revenue'].mean().to_dict()
ym_cogs_lookup = train_full.groupby('ym')['COGS'].mean().to_dict()
rev_roll_map   = dict(zip(train_full['Date'], train_full['rev_residual']))


def add_lag_features(df, rev_map, cogs_map, ym_rev, ym_cogs):
    df = df.copy()
    lg   = lambda d,n,mp: mp.get(d - pd.Timedelta(days=n), 0.0)
    lgym = lambda d,n,mp: mp.get((d - pd.Timedelta(days=n)).strftime('%Y-%m'), 0.0)
    df['rev_lag_365']              = df['Date'].apply(lambda d: lg(d, 365, rev_map))
    df['rev_lag_372']              = df['Date'].apply(lambda d: lg(d, 372, rev_map))
    df['cogs_lag_365']             = df['Date'].apply(lambda d: lg(d, 365, cogs_map))
    df['cogs_lag_372']             = df['Date'].apply(lambda d: lg(d, 372, cogs_map))
    df['rev_monthly_mean_lag365']  = df['Date'].apply(lambda d: lgym(d, 365, ym_rev))
    df['cogs_monthly_mean_lag365'] = df['Date'].apply(lambda d: lgym(d, 365, ym_cogs))
    # Rolling: dùng 30 ngày gần nhất từ rev_map
    def rolling_stat(d, n, mp, func):
        vals = [mp.get(d - pd.Timedelta(days=i), np.nan) for i in range(1, n+1)]
        vals = [v for v in vals if not np.isnan(v)]
        return func(vals) if vals else 0.0
    df['rev_roll_mean_7']  = df['Date'].apply(lambda d: rolling_stat(d, 7,  rev_map, np.mean))
    df['rev_roll_std_7']   = df['Date'].apply(lambda d: rolling_stat(d, 7,  rev_map, np.std))
    df['rev_roll_mean_30'] = df['Date'].apply(lambda d: rolling_stat(d, 30, rev_map, np.mean))
    df['rev_roll_std_30']  = df['Date'].apply(lambda d: rolling_stat(d, 30, rev_map, np.std))
    df['cogs_roll_mean_7']  = df['Date'].apply(lambda d: rolling_stat(d, 7,  cogs_map, np.mean))
    df['cogs_roll_std_7']   = df['Date'].apply(lambda d: rolling_stat(d, 7,  cogs_map, np.std))
    df['cogs_roll_mean_30'] = df['Date'].apply(lambda d: rolling_stat(d, 30, cogs_map, np.mean))
    df['cogs_roll_std_30']  = df['Date'].apply(lambda d: rolling_stat(d, 30, cogs_map, np.std))
    return df


def predict_ensemble(df, target):
    feat_cols = REV_FEATS  if target == 'rev'  else COGS_FEATS
    weights   = W_REV      if target == 'rev'  else W_COGS
    bl_col    = 'Revenue_pred' if target == 'rev' else 'COGS_pred'
    Xf = df[feat_cols].fillna(0)
    resids = [np.clip(final_models[f'{m}_{target}'].predict(Xf), -1e9, 1e9)
              for m in ['lgbm','xgboost']]
    return np.maximum(0, df[bl_col].values + sum(r*w for r,w in zip(resids, weights)))


# Pass 1: 2023
print('Pass 1: Forecasting 2023...')
sub_2023 = test_full[test_full['Date'].dt.year == 2023].copy()
sub_2023 = add_lag_features(sub_2023, rev_resid_map, cogs_resid_map, ym_rev_lookup, ym_cogs_lookup)
pred_rev_2023  = predict_ensemble(sub_2023, 'rev')
pred_cogs_2023 = predict_ensemble(sub_2023, 'cogs')
print(f'  Rev  2023: mean={pred_rev_2023.mean():,.0f}')
print(f'  COGS 2023: mean={pred_cogs_2023.mean():,.0f}')

# Update lookup với 2023 predictions
for d, rv, cg, brv, bcg in zip(sub_2023['Date'], pred_rev_2023, pred_cogs_2023,
                                 sub_2023['Revenue_pred'].values, sub_2023['COGS_pred'].values):
    rev_resid_map[d]  = rv  - brv
    cogs_resid_map[d] = cg  - bcg
ym_23 = pd.DataFrame({'Date': sub_2023['Date'].values,
                       'Revenue': pred_rev_2023, 'COGS': pred_cogs_2023})
ym_23['ym'] = ym_23['Date'].dt.strftime('%Y-%m')
ym_rev_lookup.update(ym_23.groupby('ym')['Revenue'].mean().to_dict())
ym_cogs_lookup.update(ym_23.groupby('ym')['COGS'].mean().to_dict())

# Pass 2: 2024
print('Pass 2: Forecasting 2024...')
sub_2024 = test_full[test_full['Date'].dt.year == 2024].copy()
sub_2024 = add_lag_features(sub_2024, rev_resid_map, cogs_resid_map, ym_rev_lookup, ym_cogs_lookup)
pred_rev_2024  = predict_ensemble(sub_2024, 'rev')
pred_cogs_2024 = predict_ensemble(sub_2024, 'cogs')
print(f'  Rev  2024: mean={pred_rev_2024.mean():,.0f}')
print(f'  COGS 2024: mean={pred_cogs_2024.mean():,.0f}')
print('2-pass forecast xong ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 18. FORECAST VISUALIZATION
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Revenue & COGS Forecast — DATATHON 2026 v4', fontsize=14, fontweight='bold')

tail = sales[sales['Date'].dt.year >= 2020]

for ax, actual_col, p23, p24, color, title, w in [
    (axes[0], 'Revenue', pred_rev_2023,  pred_rev_2024,  'steelblue',
     f'Revenue | LGBM={W_REV[0]:.2f} XGB={W_REV[1]:.2f}', W_REV),
    (axes[1], 'COGS',    pred_cogs_2023, pred_cogs_2024, 'orange',
     f'COGS    | LGBM={W_COGS[0]:.2f} XGB={W_COGS[1]:.2f}', W_COGS),
]:
    ax.plot(tail['Date'], tail[actual_col], lw=0.8, alpha=0.6, color=color, label='Train actual')
    ax.plot(sub_2023['Date'], p23, lw=1.2, color='red',     label='Forecast 2023')
    ax.plot(sub_2024['Date'], p24, lw=1.2, color='darkred', label='Forecast 2024')
    ax.axvline(pd.Timestamp('2023-01-01'), color='black', ls='--', alpha=0.4)
    for md in mega_dates:
        if TEST_START <= md <= TEST_END:
            ax.axvspan(md-pd.Timedelta(days=1), md+pd.Timedelta(days=1), alpha=0.12, color='gold')
    ax.set_title(title, fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M'))
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('forecast_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# 19. SUBMISSION
# ═══════════════════════════════════════════════════════════
submission = pd.concat([
    pd.DataFrame({'Date': sub_2023['Date'].values,
                  'Revenue': pred_rev_2023, 'COGS': pred_cogs_2023}),
    pd.DataFrame({'Date': sub_2024['Date'].values,
                  'Revenue': pred_rev_2024, 'COGS': pred_cogs_2024}),
], ignore_index=True)

# Giữ đúng thứ tự sample_submission
submission = submission.set_index('Date').reindex(sample_sub['Date'].tolist()).reset_index()

# Sanity checks
assert len(submission) == len(sample_sub),                            '❌ Row count'
assert (submission['Date'].values == sample_sub['Date'].values).all(),'❌ Date order'
assert submission['Revenue'].notna().all(),                           '❌ NaN Revenue'
assert submission['COGS'].notna().all(),                              '❌ NaN COGS'
assert (submission['Revenue'] >= 0).all(),                            '❌ Revenue âm'
assert (submission['COGS']    >= 0).all(),                            '❌ COGS âm'

submission['Date']    = submission['Date'].dt.strftime('%Y-%m-%d')
submission['Revenue'] = submission['Revenue'].round(2)
submission['COGS']    = submission['COGS'].round(2)
submission[['Date','Revenue','COGS']].to_csv('submission.csv', index=False)

print('=== submission.csv saved ✅ ===')
print(f'Rows: {len(submission)}')
print(submission.head(5).to_string(index=False))

meta = {
    'version'         : 'v4',
    'approach'        : 'Baseline-Residual + 2-Pass Forecast',
    'models'          : ['LightGBM (regression)', 'XGBoost (squarederror)'],
    'W_REV'           : W_REV.tolist(),
    'W_COGS'          : W_COGS.tolist(),
    'n_rev_features'  : len(REV_FEATS),
    'n_cogs_features' : len(COGS_FEATS),
    'val_r2_rev'      : round(m_ens_rev['R2'],   4),
    'val_mae_rev'     : round(m_ens_rev['MAE'],  0),
    'val_r2_cogs'     : round(m_ens_cogs['R2'],  4),
    'val_mae_cogs'    : round(m_ens_cogs['MAE'], 0),
}
with open('model_metadata.json','w') as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))